# M5 Forecasting — ETL Pipeline: **Transform Phase**

**Project:** Retail-Demand-Forecasting
**Stage:** `02_transform` — discovery, validation, and demonstration only
**Author:** Data Engineering / Analytics Team

---

## Purpose of this notebook

This notebook implements the **discovery and validation** work for the Transform phase of the
M5 Forecasting ETL pipeline. It answers the question *"what transformations does this data
actually need, and are they safe to perform?"* — before any of it is written as production
code.

Concretely, the notebook:

1. Loads the raw data straight from `data/raw/` using `pathlib`.
2. Re-verifies data integrity (this notebook does not assume the Extract-phase notebook was
   run first, or that its findings still hold).
3. Works out, column by column, which dtype conversions and missing-value strategies are
   actually needed — with evidence, not guesses.
4. Demonstrates the wide → long reshape of the sales table with `pandas.melt()`.
5. Demonstrates and validates the three-way join `sales ⋈ calendar ⋈ sell_prices`.
6. Produces one final transformed DataFrame and inspects it.
7. Explains every transformation and classifies it as **ETL** vs **feature engineering**.

> ⚠️ **Scope boundary — read this first**
>
> This notebook is **exploratory and demonstrative**, not production code:
> - No functions are defined — every step is written out inline, top to bottom, so each
>   transformation can be inspected and validated in isolation.
> - No performance optimization is attempted (no chunking, no parallelism, no query
>   pushdown). The M5 raw files are small enough to explore comfortably in memory.
> - No files are written to `data/processed/` — the deliverable of *this* notebook is a
>   validated **plan**, captured in the final "Transformation Decisions" section, that a
>   later `src/transform.py` module will implement as reusable, tested functions.


## 1. Imports

Same lightweight toolset as the Extract phase: `pathlib` for portable path resolution,
`pandas`/`numpy` for everything else.


In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

print(f"pandas version: {pd.__version__}")
print(f"numpy version : {np.__version__}")


**What was done:** Imported `pathlib`, `pandas`, and `numpy`, and set display options
so wide profiling tables and joined DataFrames stay readable in this notebook.

**Why it matters:** Keeping the import list identical to the Extract-phase notebook means both
notebooks can eventually share the same environment/requirements file without surprises.

**ETL or Feature Engineering:** Neither — this is tooling setup, not a data transformation.


## 2. Locate the Project Root and Load Raw Data

We resolve the project root the same way as the Extract phase: walk upward from the current
working directory until a folder containing `data/raw/` is found. This keeps the notebook free
of hardcoded absolute paths and makes it runnable from `notebooks/`, from a CI job, or from
anywhere else inside the repo.


In [ ]:
# Walk upward from the current working directory until a folder containing "data/raw" is found.
current = Path.cwd().resolve()
project_root = None
for candidate in [current, *current.parents]:
    if (candidate / "data" / "raw").exists():
        project_root = candidate
        break

if project_root is None:
    raise FileNotFoundError(
        f"Could not locate a 'data/raw' directory above {current}. "
        "Run this notebook from inside the Retail-Demand-Forecasting project."
    )

data_raw_dir = project_root / "data" / "raw"
print(f"Project root: {project_root}")
print(f"Raw data dir: {data_raw_dir}")

calendar_df = pd.read_csv(data_raw_dir / "calendar.csv")
sell_prices_df = pd.read_csv(data_raw_dir / "sell_prices.csv")
sales_df = pd.read_csv(data_raw_dir / "sales_train_validation.csv")

print(f"\ncalendar               : {calendar_df.shape}")
print(f"sell_prices            : {sell_prices_df.shape}")
print(f"sales_train_validation : {sales_df.shape}")


**What was done:** Located the project root via a `data/raw` marker search (no
absolute paths, no dependency on a prior notebook run), then loaded all three raw CSVs with
`pandas.read_csv()` and default settings — no dtypes, no parsing, nothing coerced yet.

**Why it matters:** This notebook needs to stand on its own: it re-derives the project root
and reloads raw data independently, so its integrity checks reflect the *current* state of
`data/raw/`, not cached assumptions carried over from the Extract-phase notebook.

**ETL or Feature Engineering:** **ETL (Extract)** — this step is technically a repeat of
extraction, kept here only so the Transform notebook is self-contained and reproducible on its
own.


## 3. Verify Data Integrity Before Transformation

Before deciding *how* to transform anything, we re-confirm the raw data is in the state we
expect: correct shapes, no unexpected fully-duplicated rows, and no missing values in the
columns that must be complete for a join to work correctly (join keys should never be null).


In [ ]:
print("--- shapes ---")
print(f"calendar               : {calendar_df.shape}")
print(f"sell_prices             : {sell_prices_df.shape}")
print(f"sales_train_validation  : {sales_df.shape}")

print("\n--- fully duplicated rows ---")
print(f"calendar               : {calendar_df.duplicated().sum()}")
print(f"sell_prices             : {sell_prices_df.duplicated().sum()}")
print(f"sales_train_validation  : {sales_df.duplicated().sum()}")

print("\n--- nulls in critical join-key columns ---")
print(f"calendar.d nulls              : {calendar_df['d'].isna().sum()}")
print(f"calendar.wm_yr_wk nulls       : {calendar_df['wm_yr_wk'].isna().sum()}")
print(f"sell_prices.wm_yr_wk nulls    : {sell_prices_df['wm_yr_wk'].isna().sum()}")
print(f"sell_prices.store_id nulls    : {sell_prices_df['store_id'].isna().sum()}")
print(f"sell_prices.item_id nulls     : {sell_prices_df['item_id'].isna().sum()}")
print(f"sales_train_validation.id nulls        : {sales_df['id'].isna().sum()}")
print(f"sales_train_validation.item_id nulls   : {sales_df['item_id'].isna().sum()}")
print(f"sales_train_validation.store_id nulls  : {sales_df['store_id'].isna().sum()}")

print("\n--- key uniqueness ---")
print(f"calendar.d is unique                         : {calendar_df['d'].is_unique}")
print(f"sales_train_validation.id is unique          : {sales_df['id'].is_unique}")
composite_dupes = sell_prices_df.duplicated(subset=["store_id", "item_id", "wm_yr_wk"]).sum()
print(f"sell_prices (store_id, item_id, wm_yr_wk) duplicated combos: {composite_dupes}")


**What was done:** Re-checked shapes, fully-duplicated rows, null counts in every
column that will act as a join key, and uniqueness of `calendar.d`, `sales.id`, and the
`sell_prices` composite key.

**Why it matters:** A join key with even a single null or an unexpected duplicate silently
changes join behavior — nulls never match in an inner/left merge on that column, and
duplicates cause row fan-out. Catching this *before* attempting any join is what separates a
validated pipeline from one that produces a quietly wrong row count.

**ETL or Feature Engineering:** **ETL** — integrity validation is a core, mandatory ETL
responsibility; it happens every run, not just during modeling exploration.


## 4. Deciding Which Columns Require Datatype Conversion

We inspect the dtype `pandas` inferred for every column against what that column *actually*
represents, to decide where a conversion is warranted. We are only **deciding** here — no
column is cast in this notebook.


In [ ]:
print("--- calendar dtypes ---")
print(calendar_df.dtypes)

print("\n--- sell_prices dtypes ---")
print(sell_prices_df.dtypes)

print("\n--- sales_train_validation dtypes (id columns + first few day columns) ---")
id_cols = ["id", "item_id", "dept_id", "cat_id", "store_id", "state_id"]
day_cols_preview = [c for c in sales_df.columns if c.startswith("d_")][:5]
print(sales_df[id_cols + day_cols_preview].dtypes)

print("\n--- distinct values for low-cardinality candidate 'category' columns ---")
for col in ["weekday", "event_type_1", "event_type_2"]:
    print(f"{col}: {calendar_df[col].nunique(dropna=True)} unique values -> {calendar_df[col].dropna().unique().tolist()}")
for col in ["cat_id", "dept_id", "state_id", "store_id"]:
    print(f"{col}: {sales_df[col].nunique()} unique values")


**What was done:** Printed the inferred dtype of every column across all three tables,
then measured cardinality of the columns that look like natural categorical candidates
(`weekday`, `event_type_1/2`, `cat_id`, `dept_id`, `state_id`, `store_id`).

**Why it matters:** `pandas` defaults every text column to generic `object` dtype, which is
memory-heavy and slow to group/filter on. Columns with a small, fixed set of repeated values
are strong candidates for `category` dtype; the day columns (`d_1..d_n`) are integer counts
that can safely be a smaller integer width than the 64-bit default; `date` is a string that
should become `datetime64`.

**Datatype conversion decisions (to be applied, not yet applied here):**

| Column(s) | Current dtype | Target dtype | Reasoning |
|---|---|---|---|
| `calendar.date` | `object` (string) | `datetime64[ns]` | Enables date arithmetic, resampling, and correct chronological sorting. |
| `calendar.weekday`, `event_type_1`, `event_type_2` | `object` | `category` | Small, fixed, repeated value sets. |
| `sales.cat_id`, `dept_id`, `state_id`, `store_id`, `item_id` | `object` | `category` | Low-to-moderate cardinality, repeated across many rows — large memory win. |
| `sales.d_1..d_n` (units sold) | `int64` | smaller int (e.g. `int16`/`int32`, sized against the observed max) | Units sold per day is a small non-negative integer; 64-bit is unnecessary. |
| `sell_prices.sell_price` | `float64` | `float32` (candidate) | Prices don't need double precision; halves memory for the largest table. |
| `calendar.wm_yr_wk`, `sell_prices.wm_yr_wk` | `int64` | left as integer (join key) | Kept as a plain integer since it is used purely as a join key, not for arithmetic. |

**ETL or Feature Engineering:** **ETL** — dtype optimization is a structural/storage decision
made once per load, independent of any specific model; it belongs in Transform, not in a
model-specific feature pipeline.


## 5. Deciding How Missing Values Should Be Handled

We quantify missing values per column, then reason about the *correct* handling for each
column individually — "how should nulls be handled" is not a single pipeline-wide rule, it
depends on what the null represents in each column.


In [ ]:
print("--- calendar missing values ---")
cal_missing = calendar_df.isna().sum()
print(cal_missing[cal_missing > 0])

print("\n--- sell_prices missing values ---")
price_missing = sell_prices_df.isna().sum()
print(price_missing[price_missing > 0] if price_missing.sum() > 0 else "None")

print("\n--- sales_train_validation missing values ---")
sales_missing = sales_df.isna().sum()
print(sales_missing[sales_missing > 0] if sales_missing.sum() > 0 else "None")


**What was done:** Counted missing values per column in all three raw tables.

**Why it matters:** In `calendar`, nulls in `event_name_1/2` and `event_type_1/2` are expected
— most days simply have no event, so a null there is **meaningful information** ("no event"),
not a defect. In `sell_prices`, a missing `sell_price` would mean *"this item/store/week
combination was never actually offered for sale"* — such rows should not silently join into
the sales fact table as if a price existed. `sales_train_validation` is not expected to contain
nulls in its `d_i` columns under the M5 schema (a day with 0 units sold is recorded as `0`, not
`NaN`) — any null found there would be a genuine data-quality issue to flag, not something to
silently fill.

**Missing value handling decisions:**

| Column(s) | Expected nulls? | Handling decision |
|---|---|---|
| `calendar.event_name_1`, `event_type_1`, `event_name_2`, `event_type_2` | Yes, majority of rows | Fill with an explicit sentinel category, e.g. `"none"` — preserves the "no event" signal as a usable categorical value instead of `NaN`. |
| `sell_prices.sell_price` | No (or rare) | If present: **do not fill** — a missing price means "not sold that week," so those rows should be **excluded** from the price join rather than imputed with a guessed value. |
| `sales.d_1..d_n` | No | If present: **investigate**, do not blindly fill — could indicate a store-opening gap or an extraction defect; the correct fill (0 vs. leave as missing vs. drop) depends on the cause. |
| `sales.item_id`, `dept_id`, `cat_id`, `store_id`, `state_id`, `id` | No (identifiers) | If present: **reject the row** — a null identifier breaks every downstream join and should never be silently filled. |

**ETL or Feature Engineering:** **Mixed.** Filling `event_*` nulls with a `"none"` sentinel is
an **ETL** decision (it's a structural cleanup applied identically every run). Deciding whether
a missing price implies "discontinued item" or engineering a `days_since_last_sale` style
signal from such gaps would be **feature engineering**, deferred to a modeling-specific
notebook.


## 6. Analyzing Duplicate Records

Beyond the simple duplicate counts in §3, we check duplication **on the natural key** of each
table (not just full-row duplication), since two rows can differ in only one incidental column
while still representing the same real-world record.


In [ ]:
print("--- sales_train_validation: duplicate 'id' values ---")
print(sales_df["id"].duplicated().sum())

print("\n--- sell_prices: duplicate (store_id, item_id, wm_yr_wk) combinations ---")
print(sell_prices_df.duplicated(subset=["store_id", "item_id", "wm_yr_wk"]).sum())

print("\n--- calendar: duplicate 'd' values ---")
print(calendar_df["d"].duplicated().sum())

print("\n--- calendar: duplicate 'date' values ---")
print(calendar_df["date"].duplicated().sum())


**What was done:** Checked for duplicates on each table's natural key
(`sales.id`, `sell_prices` composite `(store_id, item_id, wm_yr_wk)`, `calendar.d`,
`calendar.date`) rather than only full-row duplication.

**Why it matters:** Full-row duplicate counts (as checked in §3) miss "near duplicates" — two
rows with the same key but a different value in a non-key column (e.g. two different prices
recorded for the same store/item/week). Key-level duplication is what actually breaks a merge
(fan-out), so it needs to be checked independently of full-row duplication.

**ETL or Feature Engineering:** **ETL** — duplicate detection on natural keys is a mandatory
pre-join integrity gate, not a modeling concern.


## 7. Investigating the Wide-Format Sales Table

`sales_train_validation` stores one row per (item, store) series and one **column** per day
(`d_1`, `d_2`, …). Before reshaping it, we look at its structure directly: how many identifier
columns vs. day columns it has, and what a single row actually looks like across time.


In [ ]:
id_cols = ["id", "item_id", "dept_id", "cat_id", "store_id", "state_id"]
day_cols = [c for c in sales_df.columns if c.startswith("d_")]

print(f"Identifier columns ({len(id_cols)}): {id_cols}")
print(f"Day columns ({len(day_cols)}): {day_cols[0]} ... {day_cols[-1]}")
print(f"\nTotal columns: {sales_df.shape[1]} = {len(id_cols)} id columns + {len(day_cols)} day columns")

print("\n--- one full row, transposed, to see a single series across time ---")
single_series = sales_df.iloc[[0]]
print(single_series[id_cols].to_string(index=False))
print("\nFirst 10 days of that series:")
print(single_series[day_cols[:10]].T.rename(columns={0: "units_sold"}))


**What was done:** Split the sales table's columns into identifier columns vs. `d_i`
day columns, confirmed the split accounts for all columns, and inspected one full row
transposed to see what a single item/store time series actually looks like across the first
few days.

**Why it matters:** This is the direct, visual confirmation of *why* the table is called
"wide": each row already **is** a complete time series, just laid out horizontally. That layout
is efficient for storage but cannot be indexed by date, resampled, grouped by month, or merged
against `calendar`/`sell_prices` without first being reshaped — every one of those operations
expects `date` (or `d`) to be a **column value**, not a **column name**.

**ETL or Feature Engineering:** **ETL** — understanding the current physical layout is a
prerequisite investigation for the reshape decision made next; it isn't a transformation by
itself.


## 8. Converting the Sales Table from Wide to Long with `pandas.melt()`

We now demonstrate the reshape itself. `id_vars` holds the identifier columns fixed per row;
`value_vars` are the `d_i` columns to unpivot; `var_name`/`value_name` control what the new
"day key" and "units sold" columns are called.


In [ ]:
sales_long = pd.melt(
    sales_df,
    id_vars=id_cols,
    value_vars=day_cols,
    var_name="d",
    value_name="sales",
)

print(f"Wide shape : {sales_df.shape}")
print(f"Long shape : {sales_long.shape}")
print(f"Expected long rows = wide rows * day columns = {sales_df.shape[0]} * {len(day_cols)} = {sales_df.shape[0] * len(day_cols)}")
print(f"Row count matches expectation: {sales_long.shape[0] == sales_df.shape[0] * len(day_cols)}")

display(sales_long.head(10))
display(sales_long.tail(5))


**What was done:** Called `pd.melt()` on `sales_df`, holding the six identifier columns
fixed (`id_vars`) and unpivoting every `d_i` column into two new columns: `d` (the day key) and
`sales` (units sold that day). Verified the resulting row count exactly equals
`wide_rows × number_of_day_columns`, confirming no rows were silently dropped or duplicated by
the reshape.

**Why it matters:** This is the single most important structural transformation in the whole
Transform phase — every subsequent join, every time-series feature, and every model input
depends on sales data being in this one-row-per-(series, day) long format.

**ETL or Feature Engineering:** **ETL** — reshaping to long format is a structural
prerequisite required before the data is even usable, not an engineered predictive feature.


## 9. Why Long Format Is Preferable for Forecasting Pipelines

- **Joinability**: `calendar` and `sell_prices` are both keyed at daily/weekly grain with a
  `date`/`d`/`wm_yr_wk` **column**. A long `sales_long` table can merge directly on `d`; a wide
  table cannot be merged against a per-day table at all without first being reshaped.
- **Tidy-data principle**: each row now represents one *observation* (one item, one store, one
  day, one sales count) and each column represents one *variable* — the standard shape assumed
  by essentially every `pandas`/`scikit-learn`/time-series library.
- **Native time-series tooling**: operations like `groupby("id").resample()`,
  rolling/lag features (`shift()`, `rolling()`), and per-series train/validation splitting by
  date all require `date` to be an actual column (or index) with one row per time step — not a
  column name.
- **Extensibility**: adding a new day of sales in wide format means adding a new **column**
  (a schema change); in long format it means adding new **rows** (no schema change) — long
  format is far more natural for an incrementally-refreshed pipeline.
- **Memory during modeling**: most gradient-boosting/tree-based forecasting approaches (the
  common choice for M5-style intermittent demand) expect one training row per
  (series, timestep, feature-set) — i.e. long format is the format the model itself consumes.

**Trade-off worth noting:** long format is **less memory-efficient at rest** — it multiplies
row count by the number of days (see the memory-projection discussion in the Extract-phase
notebook) — but that cost is unavoidable and outweighed by the fact that the data must be in
this shape to be usable at all.

**ETL or Feature Engineering:** This section is an **explanation**, not a transformation — but
the reshape decision it justifies is squarely **ETL**.


## 10. Joining `sales` → `calendar` → `sell_prices`

We join in two steps, in the order dictated by the key structure discovered above:

1. `sales_long` ⋈ `calendar` on `d` (adds `date`, `wm_yr_wk`, weekday/month/year, event flags,
   SNAP flags).
2. result ⋈ `sell_prices` on `(store_id, item_id, wm_yr_wk)` (adds `sell_price`).

We use `how="left"` for both joins, keeping every sales row even if a matching calendar/price
row is unexpectedly absent — any such gap will show up as a new null after the join and needs
to be investigated (§11), not silently absorbed by an inner join.


In [ ]:
print(f"Rows before any join (sales_long) : {sales_long.shape[0]:,}")

# Step 1: sales_long + calendar on 'd'
sales_calendar = sales_long.merge(calendar_df, on="d", how="left")
print(f"Rows after joining calendar          : {sales_calendar.shape[0]:,}")

# Step 2: + sell_prices on (store_id, item_id, wm_yr_wk)
sales_full = sales_calendar.merge(
    sell_prices_df,
    on=["store_id", "item_id", "wm_yr_wk"],
    how="left",
)
print(f"Rows after joining sell_prices       : {sales_full.shape[0]:,}")

display(sales_full.head(10))


**What was done:** Performed the two merges in sequence with `how="left"`, printing the
row count after each step, and displayed the resulting combined table.

**Why it matters:** Doing the merges as two explicit, separately-inspected steps (rather than
one chained call) makes it possible to attribute any row-count or null change to a *specific*
join — essential for debugging when something doesn't line up.

**ETL or Feature Engineering:** **ETL** — joining the fact table to its dimension/reference
tables is a core Transform-phase responsibility; the *columns this produces* (e.g. `sell_price`,
`event_type_1`) become raw material that a later feature-engineering step might difference,
lag, or one-hot encode — but the join itself is ETL.


## 11. Validating the Joins: Row Counts and Missing Values

A `how="left"` merge never drops rows from the left table, but it can introduce **new nulls**
wherever no match was found on the right. We check both: that row count is exactly preserved
across both merges, and where (if anywhere) new nulls appeared.


In [ ]:
print("--- row count preservation ---")
print(f"sales_long rows      : {sales_long.shape[0]:,}")
print(f"sales_calendar rows  : {sales_calendar.shape[0]:,}")
print(f"sales_full rows      : {sales_full.shape[0]:,}")
print(f"Row count preserved through both merges: {sales_long.shape[0] == sales_calendar.shape[0] == sales_full.shape[0]}")

print("\n--- new nulls introduced by the calendar join (columns that came from calendar) ---")
calendar_added_cols = [c for c in calendar_df.columns if c != "d"]
print(sales_calendar[calendar_added_cols].isna().sum())

print("\n--- new nulls introduced by the sell_prices join (sell_price) ---")
print(sales_full["sell_price"].isna().sum(), "missing sell_price rows out of", len(sales_full))
missing_price_rate = sales_full["sell_price"].isna().mean()
print(f"missing sell_price rate: {missing_price_rate:.2%}")


**What was done:** Confirmed the row count is identical across `sales_long` →
`sales_calendar` → `sales_full` (proving neither `how="left"` merge fanned rows out or dropped
any), then measured how many nulls each join introduced in the columns it contributed.

**Why it matters:** A stable row count through both merges is the primary evidence that the
join keys behaved as expected (one match per left row, not zero or many). Nulls in the
`calendar`-sourced columns would mean some `d` values in `sales_long` don't exist in
`calendar` — a real integrity problem, since every sales row should have a calendar date.
Nulls in `sell_price`, on the other hand, are **expected and meaningful**: they mean that item
was not being actively priced/sold at that store in that week (e.g. before a product launched
at a given store), not a broken join.

**ETL or Feature Engineering:** **ETL** — this validation step is what makes the join
trustworthy enough to hand off to production code; interpreting *why* `sell_price` is missing
for a given series (e.g. building a `days_until_first_sale` feature from it) would be feature
engineering.


## 12. Verifying No Duplicate Records Were Introduced by the Joins

Even with row count preserved in aggregate, it's worth directly confirming that the natural key
of the joined table — `(id, d)`, i.e. one row per series per day — is still unique after both
merges. A one-to-many match on the `calendar` or `sell_prices` side (e.g. an accidental
duplicate `wm_yr_wk` row) could in principle preserve overall row count while still duplicating
*specific* keys.


In [ ]:
dupe_keys_before = sales_long.duplicated(subset=["id", "d"]).sum()
dupe_keys_after = sales_full.duplicated(subset=["id", "d"]).sum()

print(f"Duplicate (id, d) combinations before joins : {dupe_keys_before}")
print(f"Duplicate (id, d) combinations after joins  : {dupe_keys_after}")
print(f"Join introduced no duplicate (id, d) keys    : {dupe_keys_after == 0}")

full_row_dupes = sales_full.duplicated().sum()
print(f"\nFully duplicated rows in final joined table : {full_row_dupes}")


**What was done:** Checked duplication on the `(id, d)` natural key both before and
after the joins, plus a full-row duplicate check on the final joined table.

**Why it matters:** `(id, d)` — one series, one day — is the grain the final table is supposed
to have. Confirming zero duplicates on that key (not just a stable overall row count) is the
stronger, more direct proof that both merges were true one-to-one (or many-to-one-from-the-left)
matches rather than an accidental one-to-many fan-out that happened to net out in aggregate.

**ETL or Feature Engineering:** **ETL** — grain-level duplicate verification after a join is a
mandatory data-integrity gate before the table can be trusted for any downstream use.


## 13. The Final Transformed Dataset

`sales_full` is the exploratory output of this Transform-phase notebook: one row per
`(id, d)`, carrying identifiers, the calendar attributes, and price. We inspect its final
shape, columns, dtypes, and a sample to confirm it's ready to hand off to the Load phase / a
production `transform.py` module.


In [ ]:
print(f"Final shape  : {sales_full.shape}")
print(f"\nFinal columns: {list(sales_full.columns)}")

print("\n--- dtypes ---")
print(sales_full.dtypes)

print("\n--- sample rows ---")
display(sales_full.sample(10, random_state=42))

print("\n--- final null summary ---")
final_nulls = sales_full.isna().sum()
display(final_nulls[final_nulls > 0].to_frame("null_count"))


**What was done:** Printed the final shape, full column list, dtypes, a random sample
of rows, and a summary of which columns still contain nulls in the fully joined table.

**Why it matters:** This is the checkpoint where we confirm the exploratory transformation
plan produces a coherent, analysis-ready table: one row per (series, day), enriched with time
attributes and price, with the *only* remaining nulls being the ones we already expect and
have explained (sparse event columns, sparse `sell_price` gaps) — nothing unexplained.

**ETL or Feature Engineering:** **ETL** — this is the boundary artifact of the Transform phase.
Everything past this point (lag features, rolling means, price-change indicators, encoding
categoricals for a specific model) is **feature engineering** and belongs in its own
downstream notebook/module, not in Transform.


## 14. ETL vs. Feature Engineering — Recap Table

| # | Transformation | ETL or Feature Engineering | Reasoning |
|---|---|---|---|
| 1 | Load raw CSVs via `pathlib` | ETL | Extraction, repeated here only for self-containment. |
| 2 | Integrity checks (nulls in keys, key uniqueness) | ETL | Mandatory gate before any transformation. |
| 3 | Dtype conversions (`category`, `datetime64`, smaller ints/floats) | ETL | Structural/storage decision, model-independent. |
| 4 | Fill `event_*` nulls with `"none"` sentinel | ETL | Structural cleanup applied identically every run. |
| 5 | Reject/investigate nulls in identifier or `d_i` columns | ETL | Data-quality gate, not a modeling choice. |
| 6 | Duplicate detection on natural keys | ETL | Required before any merge. |
| 7 | Wide → long reshape (`pd.melt`) | ETL | Structural prerequisite; data is unusable without it. |
| 8 | Join `sales` ⋈ `calendar` ⋈ `sell_prices` | ETL | Combines fact table with its reference/dimension data. |
| 9 | Row-count / null / duplicate-key join validation | ETL | Confirms the join is trustworthy before hand-off. |
| 10 | Interpreting missing `sell_price` as "not yet launched" and building a feature from it | Feature Engineering | Domain interpretation aimed at improving a specific model, not a structural necessity. |
| 11 | Lag features, rolling means, price-change %, calendar-event proximity features | Feature Engineering | Model-specific signal construction — explicitly out of scope for this notebook. |
| 12 | One-hot / target encoding of categoricals | Feature Engineering | Depends on the specific model family chosen downstream. |

**General rule of thumb applied throughout this notebook:** if a step is required to produce a
*correct, analysis-ready* table regardless of which model will eventually be trained, it's
**ETL**. If a step is aimed at improving a *specific* model's predictive signal, it's
**feature engineering** — and belongs in a separate notebook downstream of this one.


## 15. Transformation Decisions — Summary

This section consolidates every transformation validated above into the concrete list that
should become **production code** (e.g. `src/transform.py`), in execution order.

### 15.1 Confirmed, validated transformations (safe to productionize)

1. **Load** `calendar.csv`, `sell_prices.csv`, `sales_train_validation.csv` from `data/raw/`
   via a portable `pathlib`-based project-root resolver.
2. **Integrity-check** raw data on load: no nulls in join-key columns
   (`d`, `wm_yr_wk`, `store_id`, `item_id`, `id`), key uniqueness holds
   (`calendar.d`, `sales.id`, `sell_prices` composite key).
3. **Convert dtypes**:
   - `calendar.date` → `datetime64[ns]`
   - `calendar.weekday`, `event_type_1`, `event_type_2` → `category`
   - `sales.item_id`, `dept_id`, `cat_id`, `store_id`, `state_id` → `category`
   - `sales.d_1..d_n` → smallest safe integer type (size against observed max value)
   - `sell_prices.sell_price` → `float32`
4. **Fill missing values**:
   - `calendar.event_name_1/2`, `event_type_1/2` → sentinel `"none"`
   - Any null found in an identifier or `d_i` column → **reject/flag the row**, do not
     silently fill.
   - Any null found in `sell_price` → **leave as null** (meaningful: not sold that week), do
     not impute.
5. **Reshape** `sales_train_validation` from wide to long with
   `pd.melt(sales_df, id_vars=id_cols, value_vars=day_cols, var_name="d", value_name="sales")`.
6. **Join**, in this exact order:
   - `sales_long.merge(calendar_df, on="d", how="left")`
   - `.merge(sell_prices_df, on=["store_id", "item_id", "wm_yr_wk"], how="left")`
7. **Validate every join** by checking, immediately after each merge:
   - Row count is unchanged from the pre-merge left table.
   - No new/unexpected nulls in columns that should always match (e.g. calendar columns).
   - Zero duplicates on the `(id, d)` grain.

### 15.2 Explicitly deferred to Feature Engineering (not part of this Transform phase)

- Lag features (e.g. `sales_lag_7`, `sales_lag_28`).
- Rolling statistics (rolling mean/std of sales or price over a window).
- Price-change indicators (`sell_price` vs. previous week).
- Calendar-event proximity features ("days until next event").
- Any encoding of categorical columns tailored to a specific model family
  (one-hot, target encoding, embeddings).

### 15.3 Explicit non-goals of this notebook (by design)

- No reusable functions were defined — every step is inline and independently inspectable.
- No performance optimization was attempted (chunking, parallel joins, query pushdown).
- No files were written to `data/processed/` — that belongs to the Load phase, once this plan
  is implemented as production code.
- No feature engineering was performed — see §15.2.

---
**Next notebook in the pipeline:** `notebooks/03_load.ipynb` (writes the validated, transformed
table to `data/processed/`), followed by a separate feature-engineering notebook/module for
any model-specific signal construction.
